In [0]:
import mlflow

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Current experiment:", mlflow.get_experiment_by_name("RAG-Leave-Policy"))

In [0]:
import mlflow

experiment_name = "/Shared/Leave-Policy-GenAI-Evaluation"

experiment_id = mlflow.create_experiment(experiment_name)

mlflow.set_experiment(experiment_id=experiment_id)

print("Experiment created successfully!")
print("Experiment name:", experiment_name)
print("Experiment ID:", experiment_id)

In [0]:
import mlflow

experiment_id = "671368168749726"

mlflow.set_experiment(experiment_id=experiment_id)

experiment = mlflow.get_experiment(experiment_id)

print("Active experiment:", experiment.name)
print("Experiment ID:", experiment.experiment_id)
print("MLflow version:", mlflow.__version__)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

endpoints = list(w.serving_endpoints.list())

print("Total endpoints:", len(endpoints))

for ep in endpoints:
    print("-" * 50)
    print("Endpoint name:", ep.name)
    print("Status:", ep.state)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

response = w.serving_endpoints.query(
    name="databricks-meta-llama-3-3-70b-instruct",
    inputs={
        "messages": [
            {
                "role": "user",
                "content": "Explain a company leave policy in two sentences."
            }
        ],
        "max_tokens": 150
    }
)

print(response)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

client = w.serving_endpoints.get_open_ai_client()

response = client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        {
            "role": "user",
            "content": "Explain a company leave policy in two sentences."
        }
    ],
    max_tokens=150
)

print(response.choices[0].message.content)

In [0]:
import importlib.util

print(
    "databricks_openai installed:",
    importlib.util.find_spec("databricks_openai") is not None
)

In [0]:
%pip install databricks-openai

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks_openai import DatabricksOpenAI

client = DatabricksOpenAI()

response = client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",
    messages=[
        {
            "role": "user",
            "content": "Explain a company leave policy in two sentences."
        }
    ],
    max_tokens=150
)

print(response.choices[0].message.content)

In [0]:
leave_policy = """
COMPANY LEAVE POLICY — SAMPLE

1. Casual Leave:
Employees receive 12 casual leave days per calendar year.

2. Sick Leave:
Employees receive 10 sick leave days per calendar year.

3. Annual Leave:
Employees receive 15 annual leave days per calendar year.

4. Leave Approval:
Employees must obtain manager approval before taking planned leave.

5. Emergency Leave:
Employees should inform their manager as soon as possible
when an emergency prevents advance notice.

6. Unspecified Rules:
If a question concerns a rule not defined in this document,
the assistant must say that the policy does not specify it.
"""

print(leave_policy)

In [0]:
import mlflow
from databricks_openai import DatabricksOpenAI

mlflow.set_experiment(
    experiment_id="671368168749726"
)

client = DatabricksOpenAI()

@mlflow.trace
def leave_policy_assistant(question: str) -> str:
    response = client.chat.completions.create(
        model="databricks-meta-llama-3-3-70b-instruct",
        messages=[
            {
                "role": "system",
                "content": f"""
You are a company leave-policy assistant.

Answer using ONLY the policy below:
{leave_policy}

Rules:
- Do not invent company-specific rules.
- If the policy does not contain the answer, say:
  "The provided policy does not specify this."
- Answer clearly and concisely.
"""
            },
            {
                "role": "user",
                "content": question
            }
        ],
        max_tokens=200
    )

    return response.choices[0].message.content


question = "How many casual leave days do employees receive?"
answer = leave_policy_assistant(question)

print("Question:", question)
print("Answer:", answer)

In [0]:
import mlflow

traces = mlflow.search_traces(
    experiment_ids=["671368168749726"],
    max_results=5
)

display(traces)

In [0]:
test_questions = [
    "How many casual leave days do employees receive?",
    "How many sick leave days do employees receive?",
    "How many annual leave days do employees receive?",
    "Who must approve planned leave?",
    "What should employees do during a leave emergency?",
    "Can unused casual leave be carried forward?"
]

traced_interactions = []

for question in test_questions:
    answer = leave_policy_assistant(question)

    traced_interactions.append({
        "question": question,
        "answer": answer
    })

    print("Question:", question)
    print("Answer:", answer)
    print("-" * 60)

Now we have multiple real LLM-generated answers, and each call should have an MLflow trace because our assistant function is decorated with @mlflow.trace.

Build an evaluation dataset from the traces

In [0]:
import mlflow

traces_df = mlflow.search_traces(
    experiment_ids=["671368168749726"],
    max_results=20
)

print("Number of traces found:", len(traces_df))
display(traces_df)

Why this step?
We want to move from ordinary Python results (traced_interactions) to MLflow's recorded trace data.
Our workflow is:
1. The LLM generates answers.
2. MLflow records the interactions as traces.
3. We turn suitable traces into an evaluation dataset.
4. An LLM judge evaluates the answers.
5. We compare assistant versions and apply a regression gate.

Inspect the trace columns
Before creating our evaluation dataset, let's inspect the actual fields MLflow returned. 

In [0]:
print("Columns available:")
print(traces_df.columns.tolist())

display(traces_df[["request", "response"]].head())

Extract questions and answers from the traces

In [0]:
import json
import pandas as pd

def extract_question(request):
    if isinstance(request, dict):
        return request.get("question")

    if isinstance(request, str):
        try:
            parsed = json.loads(request)
            return parsed.get("question") if isinstance(parsed, dict) else None
        except json.JSONDecodeError:
            return None

    return None


trace_examples = traces_df[["trace_id", "request", "response"]].copy()

trace_examples["question"] = trace_examples["request"].apply(
    extract_question
)

trace_examples["answer"] = trace_examples["response"].astype(str)

eval_examples = trace_examples[
    ["trace_id", "question", "answer"]
].dropna(subset=["question", "answer"])

display(eval_examples)
print("Extracted evaluation examples:", len(eval_examples))

Check the evaluation-dataset API

In [0]:
import mlflow
import inspect

print("MLflow version:", mlflow.__version__)

print(
    "GenAI datasets module available:",
    hasattr(mlflow.genai, "datasets")
)

if hasattr(mlflow.genai, "datasets"):
    print(
        "Available dataset functions:",
        [
            name for name in dir(mlflow.genai.datasets)
            if "dataset" in name.lower()
            or "create" in name.lower()
        ]
    )

    if hasattr(mlflow.genai.datasets, "create_dataset"):
        print(
            "create_dataset signature:",
            inspect.signature(
                mlflow.genai.datasets.create_dataset
            )
        )

Create the evaluation dataset

In [0]:
import mlflow

dataset = mlflow.genai.datasets.create_dataset(
    name="workspace.default.leave_policy_trace_eval",
    experiment_id="671368168749726"
)

print("Dataset name:", dataset.name)
print("Dataset ID:", dataset.dataset_id)

In [0]:
print("Dataset:", dataset.name)
print("Dataset ID:", dataset.dataset_id)
print("Trace examples:", len(eval_examples))

display(
    eval_examples[["trace_id", "question", "answer"]]
)

In [0]:
records = [
    {
        "inputs": {
            "question": row["question"]
        },
        "outputs": {
            "answer": row["answer"]
        }
    }
    for _, row in eval_examples.iterrows()
]

dataset.merge_records(records=records)

print("Saved examples:", len(records))

we're saving the original questions and the LLM-generated answers into the persistent MLflow evaluation dataset. The answers are observations, not ground truth.

Check the MLflow judge APIs

In [0]:
import mlflow
import inspect

print("MLflow version:", mlflow.__version__)

print(
    "Available GenAI scorers:",
    [name for name in dir(mlflow.genai.scorers)
     if not name.startswith("_")]
)

print(
    "Evaluate signature:",
    inspect.signature(mlflow.genai.evaluate)
)

Configure the LLM judge

In [0]:
from mlflow.genai.scorers import Correctness
import inspect

print("Correctness scorer signature:")
print(inspect.signature(Correctness))

Prepare evaluation records

In [0]:
evaluation_records = []

for _, row in eval_examples.iterrows():
    evaluation_records.append({
        "inputs": {
            "question": row["question"],
            "policy_context": leave_policy
        },
        "outputs": {
            "answer": row["answer"]
        }
    })

print("Prepared evaluation records:", len(evaluation_records))
print("Sample record:")
print(evaluation_records[0])

Create the LLM judge

In [0]:
from mlflow.genai.scorers import Correctness
import inspect

print(inspect.getdoc(Correctness))

Why this matters: An LLM judge is another LLM that grades an answer using criteria and evidence. For our project, we want it to check whether the answer follows the leave policy, including whether it avoids inventing unspecified rules.

In [0]:
expected_responses = {
    "How many casual leave days do employees receive?":
        "Employees receive 12 casual leave days per calendar year.",

    "How many sick leave days do employees receive?":
        "Employees receive 10 sick leave days per calendar year.",

    "How many annual leave days do employees receive?":
        "Employees receive 15 annual leave days per calendar year.",

    "Who must approve planned leave?":
        "Employees must obtain manager approval before taking planned leave.",

    "What should employees do during a leave emergency?":
        "Employees should inform their manager as soon as possible when an emergency prevents advance notice.",

    "Can unused casual leave be carried forward?":
        "The provided policy does not specify whether unused casual leave can be carried forward."
}

print("Expected responses prepared:", len(expected_responses))

In [0]:
import mlflow
from mlflow.genai.scorers import Correctness

# Build evaluation examples from our traced interactions.
evaluation_data = []

for _, row in eval_examples.iterrows():
    question = row["question"]
    expected = expected_responses.get(question)

    if expected is None:
        print("Skipping question without an expectation:", question)
        continue

    evaluation_data.append({
        "inputs": {
            "question": question,
            "policy_context": leave_policy
        },
        "outputs": {
            "answer": row["answer"]
        },
        "expectations": {
            "expected_response": expected
        }
    })

print("Examples to evaluate:", len(evaluation_data))

# Ask the LLM judge to score the answers.
correctness_judge = Correctness(model="databricks")

evaluation_result = mlflow.genai.evaluate(
    data=evaluation_data,
    scorers=[correctness_judge]
)

print(evaluation_result)

In [0]:
display(evaluation_result.result_df)

Create Version 2

In [0]:
import mlflow
from databricks_openai import DatabricksOpenAI

client = DatabricksOpenAI()

mlflow.set_experiment(
    experiment_id="671368168749726"
)

@mlflow.trace
def leave_policy_assistant_v2(question: str) -> str:
    response = client.chat.completions.create(
        model="databricks-meta-llama-3-3-70b-instruct",
        messages=[
            {
                "role": "system",
                "content": f"""
You are a company leave-policy assistant.

Use ONLY the following policy:
{leave_policy}

Rules:
1. Answer the employee's exact question.
2. State only facts supported by the policy.
3. Never invent leave entitlements or approval rules.
4. If the policy does not specify the answer, explicitly say so.
5. Keep the answer concise.
"""
            },
            {
                "role": "user",
                "content": question
            }
        ],
        max_tokens=200
    )

    return response.choices[0].message.content


print(
    leave_policy_assistant_v2(
        "Can unused casual leave be carried forward?"
    )
)

What's different in Version 2?
- Version 1: Uses a simpler system prompt.
- Version 2: Has explicit rules about unsupported information and concise answers.

Evaluate Version 2

In [0]:
import mlflow
from mlflow.genai.scorers import Correctness

def predict_v2(question: str, policy_context: str) -> dict:
    answer = leave_policy_assistant_v2(question)
    return {"answer": answer}

correctness_judge = Correctness(model="databricks")

evaluation_result_v2 = mlflow.genai.evaluate(
    data=evaluation_data_v2,
    predict_fn=predict_v2,
    scorers=[correctness_judge]
)

print(evaluation_result_v2)

In [0]:
print("Version 1 previous score:",
      evaluation_result.metrics["correctness/mean"])

print("Version 2 score:",
      evaluation_result_v2.metrics["correctness/mean"])

Build a simple regression gate

In [0]:
# Set the minimum acceptable correctness score
MIN_CORRECTNESS = 0.90

# Get Version 2's evaluation score
v2_score = evaluation_result_v2.metrics["correctness/mean"]

# Regression gate
if v2_score >= MIN_CORRECTNESS:
    print(f"PASS: Version 2 score = {v2_score}")
    print("Version 2 meets the quality threshold.")
else:
    print(f"FAIL: Version 2 score = {v2_score}")
    print("Do not approve Version 2.")

Set the comparison rules

In [0]:
V1_SCORE = evaluation_result.metrics["correctness/mean"]
V2_SCORE = evaluation_result_v2.metrics["correctness/mean"]

MIN_CORRECTNESS = 0.90
MAX_ALLOWED_DROP = 0.05

score_drop = V1_SCORE - V2_SCORE

print("Version 1:", V1_SCORE)
print("Version 2:", V2_SCORE)
print("Score drop:", score_drop)


In [0]:
def predict_v1(question: str, policy_context: str) -> dict:
    answer = leave_policy_assistant(question)
    return {"answer": answer}

In [0]:
evaluation_result_v1_fair = mlflow.genai.evaluate(
    data=evaluation_data_v2,
    predict_fn=predict_v1,
    scorers=[Correctness(model="databricks")]
)

print(evaluation_result_v1_fair)

In [0]:
V1_SCORE = evaluation_result_v1_fair.metrics["correctness/mean"]
V2_SCORE = evaluation_result_v2.metrics["correctness/mean"]

MIN_CORRECTNESS = 0.90
MAX_ALLOWED_DROP = 0.05

score_drop = V1_SCORE - V2_SCORE

print(f"Version 1: {V1_SCORE:.2f}")
print(f"Version 2: {V2_SCORE:.2f}")
print(f"Score drop: {score_drop:.2f}")

if V2_SCORE < MIN_CORRECTNESS:
    print("FAIL: Minimum quality threshold not met.")
elif score_drop > MAX_ALLOWED_DROP:
    print("FAIL: Quality regression is too large.")
else:
    print("PASS: Both regression gate checks passed.")

Create the 20 test questions and expected answer

In [0]:
evaluation_cases = [
    {
        "question": "How many casual leave days do employees receive?",
        "expected": "Employees receive 12 casual leave days per calendar year."
    },
    {
        "question": "How many sick leave days do employees receive?",
        "expected": "Employees receive 10 sick leave days per calendar year."
    },
    {
        "question": "How many annual leave days do employees receive?",
        "expected": "Employees receive 15 annual leave days per calendar year."
    },
    {
        "question": "Who must approve planned leave?",
        "expected": "Employees must obtain manager approval before taking planned leave."
    },
    {
        "question": "What should employees do during a leave emergency?",
        "expected": "Employees should inform their manager as soon as possible when an emergency prevents advance notice."
    },
    {
        "question": "Can unused casual leave be carried forward?",
        "expected": "The provided policy does not specify whether unused casual leave can be carried forward."
    },
    {
        "question": "Can employees take planned leave without manager approval?",
        "expected": "Employees must obtain manager approval before taking planned leave."
    },
    {
        "question": "How many casual and sick leave days are provided in total?",
        "expected": "Employees receive 12 casual leave days and 10 sick leave days, totaling 22 days."
    },
    {
        "question": "How many casual and annual leave days are provided in total?",
        "expected": "Employees receive 12 casual leave days and 15 annual leave days, totaling 27 days."
    },
    {
        "question": "How many sick and annual leave days are provided in total?",
        "expected": "Employees receive 10 sick leave days and 15 annual leave days, totaling 25 days."
    },
    {
        "question": "Does the policy specify whether unused annual leave expires?",
        "expected": "The provided policy does not specify whether unused annual leave expires."
    },
    {
        "question": "Does the policy guarantee that a manager will approve leave?",
        "expected": "No. The policy requires manager approval but does not guarantee approval."
    },
    {
        "question": "How many maternity leave days are provided?",
        "expected": "The provided policy does not specify maternity leave entitlement."
    },
    {
        "question": "Can employees carry sick leave into the next year?",
        "expected": "The provided policy does not specify whether sick leave can be carried forward."
    },
    {
        "question": "How early must planned leave be requested?",
        "expected": "The provided policy does not specify how far in advance planned leave must be requested."
    },
    {
        "question": "What should an employee do if an emergency prevents advance notice?",
        "expected": "The employee should inform their manager as soon as possible."
    },
    {
        "question": "Are employees entitled to 20 casual leave days?",
        "expected": "No. The policy specifies 12 casual leave days per calendar year."
    },
    {
        "question": "Does the policy specify public holiday entitlements?",
        "expected": "The provided policy does not specify public holiday entitlements."
    },
    {
        "question": "Can a manager reject a planned leave request?",
        "expected": "The policy requires manager approval but does not specify rejection procedures."
    },
    {
        "question": "What is the total of casual, sick, and annual leave days?",
        "expected": "Employees receive 12 casual, 10 sick, and 15 annual leave days, totaling 37 days."
    }
]

print("Total evaluation questions:", len(evaluation_cases))

In [0]:
evaluation_data_20 = []

for case in evaluation_cases:
    evaluation_data_20.append({
        "inputs": {
            "question": case["question"],
            "policy_context": leave_policy
        },
        "expectations": {
            "expected_response": case["expected"]
        }
    })

print("Prepared examples:", len(evaluation_data_20))
print("First example:")
print(evaluation_data_20[0])

Evaluate Version 2 on all 20 questions

In [0]:
evaluation_result_v2_20 = mlflow.genai.evaluate(
    data=evaluation_data_20,
    predict_fn=predict_v2,
    scorers=[Correctness(model="databricks")]
)

print(evaluation_result_v2_20)

In [0]:
display(evaluation_result_v2_20.result_df)

In [0]:
V2_SCORE_20 = evaluation_result_v2_20.metrics["correctness/mean"]

MIN_CORRECTNESS = 0.90

print(f"Version 2 correctness: {V2_SCORE_20:.2f}")

if V2_SCORE_20 >= MIN_CORRECTNESS:
    print("PASS: Version 2 meets the minimum quality threshold.")
else:
    print("FAIL: Version 2 must not be approved.")

Evaluate Version 1 on the same 20 questions

In [0]:
evaluation_result_v1_20 = mlflow.genai.evaluate(
    data=evaluation_data_20,
    predict_fn=predict_v1,
    scorers=[Correctness(model="databricks")]
)

print(evaluation_result_v1_20)

In [0]:
V1_SCORE_20 = evaluation_result_v1_20.metrics["correctness/mean"]
V2_SCORE_20 = evaluation_result_v2_20.metrics["correctness/mean"]

print(f"Version 1: {V1_SCORE_20:.2f}")
print(f"Version 2: {V2_SCORE_20:.2f}")
print(f"Score drop: {V1_SCORE_20 - V2_SCORE_20:.2f}")